In [1]:
import pandas as pd

data = pd.read_csv('Hasil_Labelling_Data_3class.csv')
data.info()
data.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7646 entries, 0 to 7645
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   normalisasi       7646 non-null   object
 1   Score             7646 non-null   int64 
 2   Sentiment         7646 non-null   object
 3   tokenize          7646 non-null   object
 4   stopword removal  7599 non-null   object
dtypes: int64(1), object(4)
memory usage: 298.8+ KB


,normalisasi,Score,Sentiment,tokenize,stopword removal
0,serius bertanya kalau tidak ada dpr semua oran...,-3,Negatif,"['serius', 'bertanya', 'kalau', 'tidak', 'ada'...",serius dpr orang hidup
1,menari di atas penderitaan rakyat,-2,Negatif,"['menari', 'di', 'atas', 'penderitaan', 'rakyat']",menari penderitaan rakyat
2,dunia akhirat gue tidak iklas byr pajak,-2,Negatif,"['dunia', 'akhirat', 'gue', 'tidak', 'iklas', ...",dunia akhirat iklas byr pajak
3,tidak usah bayar pajak titikkkk,-3,Negatif,"['tidak', 'usah', 'bayar', 'pajak', 'titikkkk']",bayar pajak titikkkk
4,parah parah pejabat indonesiaa,-3,Negatif,"['parah', 'parah', 'pejabat', 'indonesiaa']",parah parah pejabat indonesiaa


## Normalisasi Kata Slang ke Bahasa Indonesia Baku
Mengganti semua kata slang, singkatan, dan bahasa tidak baku pada kolom `normalisasi` menjadi bahasa Indonesia formal dan baku.

In [2]:
import re

# Kamus kata slang/tidak baku -> kata baku (bahasa Indonesia formal)
kamus_slang = {
    # --- Kata ganti orang ---
    'gue': 'saya', 'gw': 'saya', 'gua': 'saya', 'gwe': 'saya', 'gu': 'saya',
    'ane': 'saya', 'aye': 'saya', 'ai': 'saya', 'sy': 'saya', 'w': 'saya',
    'lu': 'kamu', 'lo': 'kamu', 'elu': 'kamu', 'ente': 'kamu', 'u': 'kamu',
    'kau': 'kamu', 'mu': 'kamu', 'km': 'kamu', 'kmu': 'kamu',
    'kalian': 'kalian', 'mrk': 'mereka', 'mreka': 'mereka',
    
    # --- Kata kerja ---
    'bikin': 'membuat', 'bkin': 'membuat', 'dibikin': 'dibuat',
    'mikir': 'berpikir', 'mkir': 'berpikir', 'pikirin': 'pikirkan',
    'ngomong': 'berbicara', 'mengomong': 'berbicara', 'ngmng': 'berbicara',
    'ngomongin': 'membicarakan', 'omongan': 'perkataan', 'omon': 'perkataan',
    'ngitung': 'menghitung', 'itung': 'menghitung', 'diitung': 'dihitung',
    'ngeles': 'mengelak', 'ngeles': 'mengelak',
    'naikin': 'menaikkan', 'naikan': 'menaikkan', 'dinaikan': 'dinaikkan',
    'nongol': 'muncul', 'nongol': 'muncul',
    'nyalon': 'mencalonkan', 'nyaleg': 'mencalonkan',
    'nolak': 'menolak',
    'ngambil': 'mengambil',
    'ngabisin': 'menghabiskan', 'ngabisin': 'menghabiskan',
    'nemu': 'menemukan',
    'balikin': 'kembalikan', 'balikin': 'kembalikan',
    'bayarin': 'membayarkan', 'bayarin': 'membayarkan',
    'bubarin': 'bubarkan', 'bubarkn': 'bubarkan',
    'pajakin': 'pajakkan',
    'naikin': 'menaikkan',
    'suruh': 'menyuruh', 'disuruh': 'disuruh',
    'dikasih': 'diberikan', 'kasih': 'berikan', 'ngasih': 'memberikan',
    'ngegaji': 'menggaji',
    'nutup': 'menutup',
    'nanggung': 'menanggung',
    'ngakali': 'mengakali',
    'nyari': 'mencari', 'nyari': 'mencari',
    'ngantuk': 'mengantuk',
    'ngeluh': 'mengeluh',
    'ngeri': 'mengerikan',
    'nyinyir': 'mengkritik',
    'mending': 'lebih baik',
    'ngerti': 'mengerti',
    'ngapain': 'mengapa',
    'ngapai': 'mengapa', 'mengapai': 'mengapa',
    'ngecek': 'memeriksa',
    'ngebut': 'cepat',

    # --- Kata sifat/keterangan ---
    'banget': 'sekali', 'bgt': 'sekali', 'bngt': 'sekali',
    'gede': 'besar', 'gedhe': 'besar',
    'kayak': 'seperti', 'kayaknya': 'sepertinya', 'kayanya': 'sepertinya',
    'kaya': 'seperti',
    'doang': 'saja', 'doank': 'saja', 'duang': 'saja',
    'cuma': 'hanya', 'cuman': 'hanya', 'cumaa': 'hanya', 'cmn': 'hanya',
    'aja': 'saja', 'ajaa': 'saja', 'ajah': 'saja',
    'emang': 'memang', 'emg': 'memang', 'mmg': 'memang',
    'bener': 'benar', 'bner': 'benar', 'benaran': 'benar',
    'segitu': 'sebanyak itu', 'sgitu': 'sebanyak itu',
    'begitu': 'begitu', 'gitu': 'begitu', 'gituh': 'begitu',
    'begini': 'begini', 'gini': 'begini',
    'malah': 'justru',
    'makin': 'semakin',
    'capek': 'lelah', 'capek': 'lelah', 'cape': 'lelah',
    'males': 'malas', 'mager': 'malas',
    'kocak': 'lucu',
    'entar': 'nanti', 'tar': 'nanti',
    'deket': 'dekat', 'dket': 'dekat',
    'bosen': 'bosan',
    'kesel': 'kesal',
    'pusing': 'bingung',
    'keren': 'hebat',
    'mantap': 'hebat', 'mantab': 'hebat',
    'enteng': 'ringan',
    'lumayan': 'cukup baik',
    'ribet': 'rumit',
    'kelar': 'selesai',
    'serem': 'menakutkan',
    'bego': 'bodoh', 'bego': 'bodoh', 'tolol': 'bodoh', 'goblok': 'bodoh',
    'oon': 'bodoh',
    'rakus': 'tamak',
    'puyeng': 'pusing',
    'apaan': 'apa',
    'gimana': 'bagaimana', 'gmn': 'bagaimana', 'gmna': 'bagaimana',
    'kenapa': 'mengapa', 'knapa': 'mengapa', 'knp': 'mengapa',
    'dimana': 'di mana', 'dmn': 'di mana', 'dimn': 'di mana',
    'disini': 'di sini', 'dsini': 'di sini',

    # --- Konjungsi / Kata hubung ---
    'tapi': 'tetapi', 'tp': 'tetapi', 'tpi': 'tetapi',
    'soalnya': 'karena', 'soale': 'karena',
    'makanya': 'oleh karena itu',
    'padahal': 'padahal',
    'lagian': 'lagipula',
    'sementara': 'sementara',
    'supaya': 'agar',
    'biar': 'agar', 'biyar': 'agar',

    # --- Kata keterangan waktu ---
    'udah': 'sudah', 'udh': 'sudah', 'dah': 'sudah', 'sdh': 'sudah',
    'blm': 'belum', 'blom': 'belum',
    'lagi': 'sedang', 'lg': 'sedang',
    'bakal': 'akan',
    'trus': 'kemudian', 'trs': 'kemudian', 'terus': 'kemudian',
    'dulu': 'dahulu',
    'kemarin': 'kemarin',
    'setaun': 'setahun',
    'mulu': 'terus menerus',

    # --- Singkatan umum ---
    'yg': 'yang', 'yng': 'yang',
    'dgn': 'dengan', 'dngn': 'dengan', 'dng': 'dengan', 'dg': 'dengan',
    'utk': 'untuk', 'untk': 'untuk', 'tuk': 'untuk', 'buat': 'untuk',
    'dr': 'dari',
    'sm': 'sama', 'sma': 'sama',
    'jg': 'juga', 'jga': 'juga',
    'pd': 'pada',
    'hrs': 'harus', 'hrus': 'harus',
    'jd': 'jadi', 'jdi': 'jadi',
    'krn': 'karena', 'karna': 'karena', 'krna': 'karena',
    'klo': 'kalau', 'kalo': 'kalau', 'klaw': 'kalau',
    'sampe': 'sampai', 'smpe': 'sampai', 'smpai': 'sampai',
    'dll': 'dan lain lain',
    'dsb': 'dan sebagainya',
    'rp': 'rupiah',
    'jt': 'juta', 'jta': 'juta', 'jtbln': 'juta per bulan', 'jtbulan': 'juta per bulan',
    'rb': 'ribu', 'rbu': 'ribu', 'rbbln': 'ribu per bulan', 'rbbulan': 'ribu per bulan',
    'byr': 'bayar', 'byar': 'bayar',
    'krja': 'kerja', 'krj': 'kerja', 'krjanya': 'kerjanya',
    'gji': 'gaji', 'gajih': 'gaji',
    'mkan': 'makan', 'mkn': 'makan', 'makn': 'makan',
    'iklas': 'ikhlas', 'ikhlas': 'ikhlas',
    'mentri': 'menteri', 'mntri': 'menteri',
    'milyar': 'miliar', 'milyaran': 'miliaran',
    'vidio': 'video', 'vedio': 'video',
    'komen': 'komentar',
    'bpk': 'bapak', 'bpk': 'bapak',
    'rkyt': 'rakyat',
    'angota': 'anggota',
    'tunjang': 'tunjangan', 'tunjangn': 'tunjangan', 'tujangan': 'tunjangan',
    'hutang': 'utang',
    'bukti': 'bukti', 'buktinya': 'buktinya',
    'harusnya': 'seharusnya',
    'pantesan': 'pantas', 'pantasan': 'pantas',
    'walopun': 'walaupun',
    'mngkn': 'mungkin',
    'pengin': 'ingin', 'pgn': 'ingin', 'pngen': 'ingin', 'pingin': 'ingin',
    'tau': 'tahu', 'tw': 'tahu',
    'pake': 'pakai', 'pke': 'pakai',
    'rame': 'ramai',
    'duit': 'uang', 'duitnya': 'uangnya',
    'aktifitas': 'aktivitas',
    'rubah': 'ubah',
    'sodara': 'saudara',
    'undangundang': 'undang undang',
    'diatas': 'di atas',
    'dibawah': 'di bawah', 'kebawah': 'ke bawah',
    'kedepan': 'ke depan',
    'ditengah': 'di tengah',
    'statment': 'pernyataan',
    'akalan': 'akal akalan',
    'okt': 'oktober',
    'buzzer': 'pendengung', 'buzer': 'pendengung',
    'mesti': 'harus',
    'percuma': 'sia sia',

    # --- Kata sapaan informal ---
    'bro': '', 'broo': '', 'brooo': '',
    'coy': '', 'cok': '', 'cuk': '',
    'woy': '', 'woi': '', 'woii': '',
    'rek': '', 'rek': '',
    'mbak': '', 'neng': '', 'om': '',
    'gan': '', 'sis': '',

    # --- Interjeksi/partikel (dihapus) ---
    'wkwk': '', 'wkwkwk': '', 'wkwkwkwk': '', 'wkwkw': '',
    'haha': '', 'hahaha': '', 'hahahaha': '',
    'hehe': '', 'hihi': '', 'huhu': '',
    'lol': '', 'xixi': '',
    'anjir': '', 'anjirr': '', 'anjjj': '', 'jir': '', 'bjir': '',
    'anjing': '', 'buset': '', 'bset': '',
    'preet': '',
    'aduh': '', 'duh': '', 'waduh': '', 'astaga': '', 'astaghfirullah': '',
    'walah': '',
    'hei': '', 'hey': '',
    'eh': '', 'oh': '', 'ah': '', 'ih': '', 'uh': '',
    'hmm': '', 'hm': '',
    'wow': '', 'wah': '',
    'lah': '', 'sih': '', 'deh': '', 'dong': '', 'kok': '',
    'nah': '', 'mah': '', 'toh': '',
    'loh': '', 'lho': '',
    'nih': '', 'tuh': '',
    'kan': '',
    'kah': '',
    'pun': '',
    'ye': '', 'yah': '',
    'iya': '', 'iyaa': '', 'iyaaa': '',

    # --- Kata bahasa daerah (Jawa, Sunda, dll) ---
    'piye': 'bagaimana', 'kumaha': 'bagaimana',
    'ora': 'tidak', 'kagak': 'tidak', 'ogah': 'tidak mau', 'ga': 'tidak', 'gak': 'tidak',
    'ngga': 'tidak', 'nggak': 'tidak', 'kagak': 'tidak', 'gk': 'tidak', 'g': 'tidak',
    'wong': 'orang', 'bocah': 'anak', 'bocil': 'anak kecil',
    'iki': 'ini', 'iku': 'itu',
    'kabeh': 'semua',
    'seng': 'yang',
    'karo': 'dengan',
    'mbok': 'tolong',
    'dewe': 'sendiri',
    'mangan': 'makan',
    'bae': 'saja',
    'naon': 'apa',
    'teu': 'tidak',
    'imah': 'rumah',
    'blas': 'sama sekali',
    'rada': 'agak',

    # --- Penulisan tidak baku lainnya ---
    'indonesiaa': 'indonesia',
    'titikkkk': 'titik',
    'pakk': 'pak',
    'indonesiaa': 'indonesia',
}

def normalisasi_slang(teks):
    """Mengganti kata slang/tidak baku menjadi kata baku"""
    if not isinstance(teks, str):
        return teks
    
    # Hapus karakter berulang berlebihan (misal: parahhh -> parah, baguuus -> bagus)
    teks = re.sub(r'(.)\1{2,}', r'\1', teks)
    
    kata_list = teks.split()
    hasil = []
    for kata in kata_list:
        kata_baku = kamus_slang.get(kata, kata)
        if kata_baku != '':  # Hapus kata yang dipetakan ke string kosong
            hasil.append(kata_baku)
    return ' '.join(hasil)

# Terapkan normalisasi pada kolom 'normalisasi'
data['normalisasi'] = data['normalisasi'].apply(normalisasi_slang)

# Tampilkan sampel hasil
print(f"Jumlah data: {len(data)}")
data[['normalisasi', 'Sentiment']].head(10)

Jumlah data: 7646


,normalisasi,Sentiment
0,serius bertanya kalau tidak ada dpr semua oran...,Negatif
1,menari di atas penderitaan rakyat,Negatif
2,dunia akhirat saya tidak ikhlas bayar pajak,Negatif
3,tidak usah bayar pajak titik,Negatif
4,parah parah pejabat indonesia,Negatif
5,rakyat sengsara anggota dpr bahagia,Negatif
6,bantuan listrik sumpah demi allah sakit hati d...,Negatif
7,demi allah saya tidak ikhlas dunia akhirat gaj...,Negatif
8,ini orang orang prabowo pejabat semakin sepert...,Negatif
9,pantas pada rebutan kursi dpr sampai nyuap rak...,Negatif


In [3]:
# Re-tokenize setelah normalisasi
data['tokenize'] = data['normalisasi'].apply(lambda x: x.split() if isinstance(x, str) else [])

# Re-apply stopword removal
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('indonesian'))

data['stopword removal'] = data['tokenize'].apply(
    lambda x: ' '.join([word for word in x if word not in stop_words])
)

# Bersihkan spasi berlebih
data['normalisasi'] = data['normalisasi'].str.strip().str.replace(r'\s+', ' ', regex=True)
data['stopword removal'] = data['stopword removal'].str.strip().str.replace(r'\s+', ' ', regex=True)

print("Contoh hasil normalisasi:")
data[['normalisasi', 'stopword removal', 'Sentiment']].head(10)

Contoh hasil normalisasi:


,normalisasi,stopword removal,Sentiment
0,serius bertanya kalau tidak ada dpr semua oran...,serius dpr orang hidup,Negatif
1,menari di atas penderitaan rakyat,menari penderitaan rakyat,Negatif
2,dunia akhirat saya tidak ikhlas bayar pajak,dunia akhirat ikhlas bayar pajak,Negatif
3,tidak usah bayar pajak titik,bayar pajak titik,Negatif
4,parah parah pejabat indonesia,parah parah pejabat indonesia,Negatif
5,rakyat sengsara anggota dpr bahagia,rakyat sengsara anggota dpr bahagia,Negatif
6,bantuan listrik sumpah demi allah sakit hati d...,bantuan listrik sumpah allah sakit hati denger...,Negatif
7,demi allah saya tidak ikhlas dunia akhirat gaj...,allah ikhlas dunia akhirat gaji dpr menaikkan ...,Negatif
8,ini orang orang prabowo pejabat semakin sepert...,orang orang prabowo pejabat rakyat sengsara,Negatif
9,pantas pada rebutan kursi dpr sampai nyuap rak...,rebutan kursi dpr nyuap rakyat nyoblos uang,Negatif


In [4]:
# Verifikasi: cek apakah masih ada kata slang yang tersisa
from collections import Counter

all_words = ' '.join(data['normalisasi'].dropna().astype(str)).split()
word_freq = Counter(all_words)

# Daftar kata slang yang perlu dicek
slang_check = ['gue', 'gua', 'gw', 'lu', 'lo', 'elo', 'bgt', 'banget', 'kayak', 
               'doang', 'cuma', 'cuman', 'emang', 'udah', 'udh', 'aja', 'bikin',
               'mikir', 'ngomong', 'duit', 'gede', 'tapi', 'kagak', 'wkwk', 'anjir',
               'byr', 'gajih', 'iklas', 'tau', 'bro', 'gila', 'anjing', 'dong',
               'sih', 'lah', 'deh', 'nih', 'tuh', 'kok', 'mah', 'kan', 'loh']

print("=== Pengecekan kata slang yang tersisa ===")
sisa = False
for kata in slang_check:
    count = word_freq.get(kata, 0)
    if count > 0:
        print(f"  '{kata}' masih ditemukan: {count} kali")
        sisa = True
        
if not sisa:
    print("  SEMUA kata slang sudah berhasil dinormalisasi!")
    
print(f"\nTotal kata unik: {len(word_freq)}")
print(f"Top 20 kata terbanyak: {word_freq.most_common(20)}")

=== Pengecekan kata slang yang tersisa ===
  'gila' masih ditemukan: 56 kali

Total kata unik: 9614
Top 20 kata terbanyak: [('yang', 2268), ('tidak', 2136), ('juta', 1809), ('dpr', 1723), ('di', 1723), ('ya', 1646), ('rakyat', 1511), ('itu', 1342), ('saja', 1264), ('untuk', 1130), ('gaji', 1017), ('sudah', 839), ('hanya', 812), ('ada', 806), ('rumah', 794), ('ini', 791), ('beras', 790), ('tunjangan', 780), ('kalau', 764), ('mereka', 689)]


In [5]:
# Simpan hasil normalisasi ke file CSV
data.to_csv('Hasil_Labelling_Data_3class.csv', encoding='utf-8', index=False)
print("Data berhasil disimpan ke 'Hasil_Labelling_Data_3class.csv'")
print(f"Jumlah baris: {len(data)}")
data.head(10)

Data berhasil disimpan ke 'Hasil_Labelling_Data_3class.csv'
Jumlah baris: 7646


,normalisasi,Score,Sentiment,tokenize,stopword removal
0,serius bertanya kalau tidak ada dpr semua oran...,-3,Negatif,"[serius, bertanya, kalau, tidak, ada, dpr, sem...",serius dpr orang hidup
1,menari di atas penderitaan rakyat,-2,Negatif,"[menari, di, atas, penderitaan, rakyat]",menari penderitaan rakyat
2,dunia akhirat saya tidak ikhlas bayar pajak,-2,Negatif,"[dunia, akhirat, saya, tidak, ikhlas, bayar, p...",dunia akhirat ikhlas bayar pajak
3,tidak usah bayar pajak titik,-3,Negatif,"[tidak, usah, bayar, pajak, titik]",bayar pajak titik
4,parah parah pejabat indonesia,-3,Negatif,"[parah, parah, pejabat, indonesia]",parah parah pejabat indonesia
5,rakyat sengsara anggota dpr bahagia,-2,Negatif,"[rakyat, sengsara, anggota, dpr, bahagia]",rakyat sengsara anggota dpr bahagia
6,bantuan listrik sumpah demi allah sakit hati d...,-4,Negatif,"[bantuan, listrik, sumpah, demi, allah, sakit,...",bantuan listrik sumpah allah sakit hati denger...
7,demi allah saya tidak ikhlas dunia akhirat gaj...,-2,Negatif,"[demi, allah, saya, tidak, ikhlas, dunia, akhi...",allah ikhlas dunia akhirat gaji dpr menaikkan ...
8,ini orang orang prabowo pejabat semakin sepert...,-1,Negatif,"[ini, orang, orang, prabowo, pejabat, semakin,...",orang orang prabowo pejabat rakyat sengsara
9,pantas pada rebutan kursi dpr sampai nyuap rak...,-4,Negatif,"[pantas, pada, rebutan, kursi, dpr, sampai, ny...",rebutan kursi dpr nyuap rakyat nyoblos uang
